In [1]:

import corner

#! /usr/bin/env python3
import numpy as np
import emcee
import sys, os
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm

from multiprocessing import Pool, current_process
import threading
import queue
import time

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

def set_atmos_run_RT(qNH3: float, 
                     T0: float = 180.0, 
                     RHmax: float=1.0,
                     adlnNH3dlnP: float=0.0,
                     pmin: float = 0.0, 
                     pmax: float = 0.0,
                     jindex: int = 0
                     ):  
    ## construct atmos with a rh limit
    ## jindex is the index of current processer, starting from zero, will add to mb.jst in canoe backend 
    mb.construct_atmosphere(pin, qNH3, T0, RHmax, jindex,"dry", 2500, 500)


    ## do radiative transfer
    rad.cal_radiance(mb, mb.k_st, mb.j_st+jindex)
    tb = np.array([0.0] * 4 * nb)
    # print(tb.shape)
    for ib in range(nb):
        toa = rad.get_band(ib).get_toa()[0]
        tb[ib * 4 : ib * 4 + 4] = toa
    return tb


# Define the forward model simulator
def fwd_simulator(xNH3,theta, adlnNH3dlnP, Pmax , RHmax):
    tb=set_atmos_run_RT(xNH3, # NH3.ppmv
                     theta,       # Temperature
                     RHmax,         # RH_max_NH3
                     adlnNH3dlnP,        # adlnNH3/dlnP
                     1E-3,         # pmin [Pa]
                     Pmax,          # pmax [Pa]
                     0)
    print(xNH3,theta, adlnNH3dlnP, Pmax, RHmax)
    print(tb)
    return tb[::4], (tb[::4]-tb[3::4])/(tb[::4])*100
    # return  Nadir tb, R45

nx2 = 5  ## shall not be less than N_walkers, can be a little greater for safty.

## initialize Canoe
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")
# print(pin.get_real("problem", "qH2O.ppmv"))

pin.set_boolean("job","verbose", False)

print(pin.get_string("mesh","nx2"))
pin.set_string("mesh","nx2", f"{nx2}")

print(pin.get_string("mesh","nx2"))

mesh = Mesh(pin)
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)
rad = mb.get_rad()
nb = rad.get_num_bands()

# Assumed parameter values and uncertainties
theta_mean, theta_std = 175.69, 5.6         # Mean and standard deviation for theta
xNH3_mean, xNH3_std = 354.96, 38.8          # Mean and standard deviation for xNH3
adlnNH3dlnP_mean, adlnNH3dlnP_std = -0.06, 0.01  # Mean and standard deviation for adlnNH3dlnP
Pmax_mean, Pmax_std = 4.88, 1         # Mean and standard deviation for Pmax
RHmax_mean, RHmax_std= 1, 0.15

# Arrays to store the results of R45 and brightness temperature

BT, R45 = fwd_simulator(xNH3_mean,theta_mean,  adlnNH3dlnP_mean, Pmax_mean, RHmax_mean)
    
print("Nominal BT:", BT)
print("Nominal R45:", R45)

# Save the results to an HDF5 file
with h5py.File('TB_R45_dry_uniform.h5', 'w') as h5file:
    h5file.create_dataset('R45', data=R45)
    h5file.create_dataset('BT', data=BT)



5Log, "2025-09-18 22:51:05",        canoe, 1., "Installing monitor canoe"

5
Log, "2025-09-18 22:51:05",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-09-18 22:51:05",         snap, 3., "Installing monitor snap"
Log, "2025-09-18 22:51:05",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-18 22:51:05",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-18 22:51:05",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-18 22:51:05",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-18 22:51:05",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-18 22:51:05",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-18 22:51:05", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-18 22:51:05", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-18 22:51:05",         harp, 9., "Installing monitor harp"
Log, "2025-09-18 22:51:05",         harp, 9.1., "Initialize Radiation"
Log, "2025-

In [1]:

import corner

#! /usr/bin/env python3
import numpy as np
import emcee
import sys, os
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm

from multiprocessing import Pool, current_process
import threading
import queue
import time

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

def set_atmos_run_RT(qNH3: float, 
                     T0: float = 180.0, 
                     RHmax: float=1.0,
                     adlnNH3dlnP: float=0.0,
                     pmin: float = 0.0, 
                     pmax: float = 0.0,
                     jindex: int = 0
                     ):  
    ## construct atmos with a rh limit
    ## jindex is the index of current processer, starting from zero, will add to mb.jst in canoe backend 
    mb.construct_atmosphere(pin, qNH3, T0, RHmax, jindex,"dry", 2500, 500)

    ## modify the top humidity with a increment
    mb.modify_dlnNH3dlnP_rhmax(pin, adlnNH3dlnP, pmin, pmax, RHmax, jindex, "dry") 


    ## do radiative transfer
    rad.cal_radiance(mb, mb.k_st, mb.j_st+jindex)
    tb = np.array([0.0] * 4 * nb)
    # print(tb.shape)
    for ib in range(nb):
        toa = rad.get_band(ib).get_toa()[0]
        tb[ib * 4 : ib * 4 + 4] = toa
    return tb


# Define the forward model simulator
def fwd_simulator(xNH3,theta, adlnNH3dlnP, Pmax , RHmax):
    tb=set_atmos_run_RT(xNH3, # NH3.ppmv
                     theta,       # Temperature
                     RHmax,         # RH_max_NH3
                     adlnNH3dlnP,        # adlnNH3/dlnP
                     1E-3,         # pmin [Pa]
                     Pmax,          # pmax [Pa]
                     0)
    print(xNH3,theta, adlnNH3dlnP, Pmax, RHmax)
    print(tb)
    return tb[::4], (tb[::4]-tb[3::4])/(tb[::4])*100
    # return  Nadir tb, R45

nx2 = 5  ## shall not be less than N_walkers, can be a little greater for safty.

## initialize Canoe
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")
# print(pin.get_real("problem", "qH2O.ppmv"))

pin.set_boolean("job","verbose", False)

print(pin.get_string("mesh","nx2"))
pin.set_string("mesh","nx2", f"{nx2}")

print(pin.get_string("mesh","nx2"))

mesh = Mesh(pin)
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)
rad = mb.get_rad()
nb = rad.get_num_bands()

# Assumed parameter values and uncertainties
theta_mean, theta_std = 175.69, 5.6         # Mean and standard deviation for theta
xNH3_mean, xNH3_std = 354.96, 38.8          # Mean and standard deviation for xNH3
adlnNH3dlnP_mean, adlnNH3dlnP_std = -0.06, 0.01  # Mean and standard deviation for adlnNH3dlnP
Pmax_mean, Pmax_std = 4.88E5, 1E5         # Mean and standard deviation for Pmax
RHmax_mean, RHmax_std= 1, 0.15

# Arrays to store the results of R45 and brightness temperature

BT, R45 = fwd_simulator(xNH3_mean,theta_mean,  adlnNH3dlnP_mean, Pmax_mean, RHmax_mean)
    
print("Nominal BT:", BT)
print("Nominal R45:", R45)

# Save the results to an HDF5 file
with h5py.File('TB_R45_dry_depletedNH3.h5', 'w') as h5file:
    h5file.create_dataset('R45', data=R45)
    h5file.create_dataset('BT', data=BT)



5Log, "2025-09-18 22:51:38",        canoe, 1., "Installing monitor canoe"

5
Log, "2025-09-18 22:51:38",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-09-18 22:51:38",         snap, 3., "Installing monitor snap"
Log, "2025-09-18 22:51:38",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-18 22:51:38",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-18 22:51:38",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-18 22:51:38",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-18 22:51:38",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-18 22:51:38",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-18 22:51:38", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-18 22:51:38", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-18 22:51:38",         harp, 9., "Installing monitor harp"
Log, "2025-09-18 22:51:38",         harp, 9.1., "Initialize Radiation"
Log, "2025-

In [1]:

import corner

#! /usr/bin/env python3
import numpy as np
import emcee
import sys, os
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm

from multiprocessing import Pool, current_process
import threading
import queue
import time

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

def set_atmos_run_RT(qNH3: float, 
                     T0: float = 180.0, 
                     RHmax: float=1.0,
                     adlnNH3dlnP: float=0.0,
                     pmin: float = 0.0, 
                     pmax: float = 0.0,
                     jindex: int = 0
                     ):  
    ## construct atmos with a rh limit
    ## jindex is the index of current processer, starting from zero, will add to mb.jst in canoe backend 
    mb.construct_atmosphere(pin, qNH3, T0, RHmax, jindex,"dry", 2500, 500)

    ## modify the top humidity with a increment
    mb.modify_dlnNH3dlnP_rhmax(pin, adlnNH3dlnP, pmin, pmax, RHmax, jindex, "dry") 


    ## do radiative transfer
    rad.cal_radiance(mb, mb.k_st, mb.j_st+jindex)
    tb = np.array([0.0] * 4 * nb)
    # print(tb.shape)
    for ib in range(nb):
        toa = rad.get_band(ib).get_toa()[0]
        tb[ib * 4 : ib * 4 + 4] = toa
    return tb


# Define the forward model simulator
def fwd_simulator(xNH3,theta, adlnNH3dlnP, Pmax , RHmax):
    tb=set_atmos_run_RT(xNH3, # NH3.ppmv
                     theta,       # Temperature
                     RHmax,         # RH_max_NH3
                     adlnNH3dlnP,        # adlnNH3/dlnP
                     1E-3,         # pmin [Pa]
                     Pmax,          # pmax [Pa]
                     0)
    print(xNH3,theta, adlnNH3dlnP, Pmax, RHmax)
    print(tb)
    return tb[::4], (tb[::4]-tb[3::4])/(tb[::4])*100
    # return  Nadir tb, R45

nx2 = 5  ## shall not be less than N_walkers, can be a little greater for safty.

## initialize Canoe
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")
# print(pin.get_real("problem", "qH2O.ppmv"))

pin.set_boolean("job","verbose", False)

print(pin.get_string("mesh","nx2"))
pin.set_string("mesh","nx2", f"{nx2}")

print(pin.get_string("mesh","nx2"))

mesh = Mesh(pin)
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)
rad = mb.get_rad()
nb = rad.get_num_bands()

# Assumed parameter values and uncertainties
theta_mean, theta_std = 175.69, 5.6         # Mean and standard deviation for theta
xNH3_mean, xNH3_std = 354.96, 38.8          # Mean and standard deviation for xNH3
adlnNH3dlnP_mean, adlnNH3dlnP_std = -0.06, 0.01  # Mean and standard deviation for adlnNH3dlnP
Pmax_mean, Pmax_std = 4.88E5, 1E5         # Mean and standard deviation for Pmax
RHmax_mean, RHmax_std= 0.71, 0.15

# Arrays to store the results of R45 and brightness temperature

BT, R45 = fwd_simulator(xNH3_mean,theta_mean,  adlnNH3dlnP_mean, Pmax_mean, RHmax_mean)
    
print("Nominal BT:", BT)
print("Nominal R45:", R45)

# Save the results to an HDF5 file
with h5py.File('TB_R45_dry_depletedNH3_rhmax.h5', 'w') as h5file:
    h5file.create_dataset('R45', data=R45)
    h5file.create_dataset('BT', data=BT)



5Log, "2025-09-18 22:52:27",        canoe, 1., "Installing monitor canoe"

5
Log, "2025-09-18 22:52:27",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-09-18 22:52:27",         snap, 3., "Installing monitor snap"
Log, "2025-09-18 22:52:27",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-18 22:52:27",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-18 22:52:27",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-18 22:52:27",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-18 22:52:27",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-18 22:52:27",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-18 22:52:27", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-18 22:52:27", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-18 22:52:27",         harp, 9., "Installing monitor harp"
Log, "2025-09-18 22:52:27",         harp, 9.1., "Initialize Radiation"
Log, "2025-

In [1]:

import corner

#! /usr/bin/env python3
import numpy as np
import emcee
import sys, os
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm

from multiprocessing import Pool, current_process
import threading
import queue
import time

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

def set_atmos_run_RT(qNH3: float, 
                     T0: float = 180.0, 
                     RHmax: float=1.0,
                     adlnNH3dlnP: float=0.0,
                     pmin: float = 0.0, 
                     pmax: float = 0.0,
                     jindex: int = 0
                     ):  
    ## construct atmos with a rh limit

    Ts=mb.retrieve_Ts_given_T1bar(pin, qNH3, T0, RHmax, jindex,"dry", 2500, 500)

    ## construct moist atmosphere with bottom ts
    mb.construct_atmosphere_Ts(pin, qNH3, Ts, RHmax, jindex,"pseudo", 1600)

    ## modify the top humidity with a increment
    # mb.modify_dlnNH3dlnP_rhmax(adlnNH3dlnP, pmin, pmax, RHmax, jindex) 

    ## do radiative transfer
    rad.cal_radiance(mb, mb.k_st, mb.j_st+jindex)
    tb = np.array([0.0] * 4 * nb)
    # print(tb.shape)
    for ib in range(nb):
        toa = rad.get_band(ib).get_toa()[0]
        tb[ib * 4 : ib * 4 + 4] = toa
    return tb


# Define the forward model simulator
def fwd_simulator(xNH3,theta, adlnNH3dlnP, Pmax, RHmax=1):
    tb=set_atmos_run_RT(xNH3, # NH3.ppmv
                     theta,       # Temperature
                     RHmax,         # RH_max_NH3
                     adlnNH3dlnP,        # adlnNH3/dlnP
                     1E-3,         # pmin [Pa]
                     Pmax,          # pmax [Pa]
                     0)
    print(xNH3,theta, adlnNH3dlnP, Pmax)
    print(tb)
    return tb[::4], (tb[::4]-tb[3::4])/(tb[::4])*100
    # return  Nadir tb, R45

nx2 = 5  ## shall not be less than N_walkers, can be a little greater for safty.

## initialize Canoe
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")
# print(pin.get_real("problem", "qH2O.ppmv"))

pin.set_boolean("job","verbose", False)

print(pin.get_string("mesh","nx2"))
pin.set_string("mesh","nx2", f"{nx2}")

print(pin.get_string("mesh","nx2"))

mesh = Mesh(pin)
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)
rad = mb.get_rad()
nb = rad.get_num_bands()

# Assumed parameter values and uncertainties
theta_mean, theta_std = 175.65, 5.56         # Mean and standard deviation for theta
xNH3_mean, xNH3_std = 355.45, 38.8          # Mean and standard deviation for xNH3


BT, R45 = fwd_simulator(xNH3_mean,theta_mean, 0,0, RHmax=1)

# Save the results to an HDF5 file
with h5py.File('TB_R45_moist_uniform.h5', 'w') as h5file:
    h5file.create_dataset('R45', data=R45)
    h5file.create_dataset('BT', data=BT)



5Log, "2025-09-18 15:50:32",        canoe, 1., "Installing monitor canoe"

5
Log, "2025-09-18 15:50:32",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-09-18 15:50:32",         snap, 3., "Installing monitor snap"
Log, "2025-09-18 15:50:32",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-18 15:50:32",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-18 15:50:32",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-18 15:50:32",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-18 15:50:32",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-18 15:50:32",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-18 15:50:32", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-18 15:50:32", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-18 15:50:32",         harp, 9., "Installing monitor harp"
Log, "2025-09-18 15:50:32",         harp, 9.1., "Initialize Radiation"
Log, "2025-

In [1]:
import corner

#! /usr/bin/env python3
import numpy as np
import emcee
import sys, os
import matplotlib.pyplot as plt
import h5py
from scipy.stats import norm

from multiprocessing import Pool, current_process
import threading
import queue
import time

sys.path.append("build/python")
sys.path.append(".")
from canoe import def_species, load_configure
from canoe.snap import def_thermo
from canoe.athena import Mesh, ParameterInput, Outputs, MeshBlock
# from canoe.harp import radiation_band, radiation

def set_atmos_run_RT(qNH3: float, 
                     T0: float = 180.0, 
                     RHmax: float=1.0,
                     adlnNH3dlnP: float=0.0,
                     pmin: float = 0.0, 
                     pmax: float = 0.0,
                     jindex: int = 0
                     ):  
    ## construct atmos with a rh limit

    Ts=mb.retrieve_Ts_given_T1bar(pin, qNH3, T0, RHmax, jindex,"dry", 2500, 500)

    ## construct moist atmosphere with bottom ts
    mb.construct_atmosphere_Ts(pin, qNH3, Ts, RHmax, jindex,"pseudo", 1600)

    ## modify the top humidity with a increment
    # mb.modify_dlnNH3dlnP_rhmax(adlnNH3dlnP, pmin, pmax, RHmax, jindex) 

    ## do radiative transfer
    rad.cal_radiance(mb, mb.k_st, mb.j_st+jindex)
    tb = np.array([0.0] * 4 * nb)
    # print(tb.shape)
    for ib in range(nb):
        toa = rad.get_band(ib).get_toa()[0]
        tb[ib * 4 : ib * 4 + 4] = toa
    return tb


# Define the forward model simulator
def fwd_simulator(xNH3,theta, adlnNH3dlnP, Pmax, RHmax=1):
    tb=set_atmos_run_RT(xNH3, # NH3.ppmv
                     theta,       # Temperature
                     RHmax,         # RH_max_NH3
                     adlnNH3dlnP,        # adlnNH3/dlnP
                     1E-3,         # pmin [Pa]
                     Pmax,          # pmax [Pa]
                     0)
    print(xNH3,theta, adlnNH3dlnP, Pmax)
    print(tb)
    return tb[::4], (tb[::4]-tb[3::4])/(tb[::4])*100
    # return  Nadir tb, R45

nx2 = 5  ## shall not be less than N_walkers, can be a little greater for safty.

## initialize Canoe
global pin
pin = ParameterInput()
pin.load_from_file("juno_mwr.inp")

vapors = pin.get_string("species", "vapor").split(", ")
clouds = pin.get_string("species", "cloud").split(", ")
tracers = pin.get_string("species", "tracer").split(", ")

def_species(vapors=vapors, clouds=clouds, tracers=tracers)
def_thermo(pin)

config = load_configure("juno_mwr.yaml")
# print(pin.get_real("problem", "qH2O.ppmv"))

pin.set_boolean("job","verbose", False)

print(pin.get_string("mesh","nx2"))
pin.set_string("mesh","nx2", f"{nx2}")

print(pin.get_string("mesh","nx2"))

mesh = Mesh(pin)
mesh.initialize(pin)

global mb, rad, nb
mb = mesh.meshblock(0)
rad = mb.get_rad()
nb = rad.get_num_bands()

# Assumed parameter values and uncertainties
theta_mean, theta_std = 175.65, 5.56         # Mean and standard deviation for theta
xNH3_mean, xNH3_std = 355.45, 38.8          # Mean and standard deviation for xNH3


BT, R45 = fwd_simulator(xNH3_mean,theta_mean, 0,0, RHmax=0.62)

# Save the results to an HDF5 file
with h5py.File('TB_R45_moist_rhmax.h5', 'w') as h5file:
    h5file.create_dataset('R45', data=R45)
    h5file.create_dataset('BT', data=BT)

5Log, "2025-09-18 15:54:26",        canoe, 1., "Installing monitor canoe"

5
Log, "2025-09-18 15:54:26",        canoe, 1.1., "Initialize IndexMap"
Log, "2025-09-18 15:54:26",         snap, 3., "Installing monitor snap"
Log, "2025-09-18 15:54:26",         snap, 3.1., "Initialize Thermodynamics"
Log, "2025-09-18 15:54:26",         snap, 3.1.1., "Enrolling vapor functions"
Log, "2025-09-18 15:54:26",         snap, 3.1.1.1., "Enrolling H2O vapor pressures"
Log, "2025-09-18 15:54:26",         snap, 3.1.2.1., "Enrolling NH3 vapor pressures"
Log, "2025-09-18 15:54:26",         snap, 4.1., "Initialize Decomposition"
Log, "2025-09-18 15:54:26",         snap, 5.1., "Initialize ImplicitSolver"
Log, "2025-09-18 15:54:26", microphysics, 7., "Installing monitor microphysics"
Log, "2025-09-18 15:54:26", microphysics, 7.1., "Initialize Microphysics"
Log, "2025-09-18 15:54:26",         harp, 9., "Installing monitor harp"
Log, "2025-09-18 15:54:26",         harp, 9.1., "Initialize Radiation"
Log, "2025-